In [1]:
pip install earthaccess xarray netCDF4 h5netcdf dask tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ===========================
# FULL MCS PIPELINE (STORAGE-SAFE)
# MERGIR -> RAW -> WNP -> 3-hourly -> MCS objects -> MCS tracks -> monthly concat
# Deletes RAW files after each successful day
# ===========================

import os
import glob
import subprocess
import calendar
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from scipy.ndimage import label

# ---------------------------
# CONFIG
# ---------------------------
CFG = {
    # raw downloaded MERGIR files
    "RAW_DIR": Path(r"C:\Users\_s2218026\Documents\lab\mergir_raw"),

    # cropped WNP files
    "WNP_DIR": Path(r"C:\Users\_s2218026\Documents\lab\mergir_wnp"),

    # final CSV outputs
    "OUT_DIR": Path(r"C:\Users\_s2218026\Documents\lab\mcs_output"),

    # WNP domain
    "LAT_MIN": 0.0,
    "LAT_MAX": 30.0,
    "LON_MIN": 100.0,
    "LON_MAX": 180.0,

    # MCS detection settings
    "TB_TH": 233.0,
    "AREA_MIN_KM2": 5000.0,

    # tracking settings
    "TRACK_MIN_IOU": 0.01,
    "TRACK_MAX_KM": 300.0,

    # MERGIR source
    "BASE_URL": "https://data.gesdisc.earthdata.nasa.gov/data/MERGED_IR/GPM_MERGIR.1",
    "COOKIES": str(Path.home() / ".urs_cookies"),
}

for k in ["RAW_DIR", "WNP_DIR", "OUT_DIR"]:
    CFG[k].mkdir(parents=True, exist_ok=True)

# ---------------------------
# DOWNLOAD HELPERS
# ---------------------------
def _doy(date):
    return f"{date.timetuple().tm_yday:03d}"

def mergir_url(date, hour, cfg=CFG):
    y = date.year
    ddd = _doy(date)
    ymdh = f"{y}{date.month:02d}{date.day:02d}{hour:02d}"
    return f"{cfg['BASE_URL']}/{y}/{ddd}/merg_{ymdh}_4km-pixel.nc4"

def download_mergir_day(date_str, cfg=CFG, verbose=True):
    date = pd.Timestamp(date_str).date()

    for h in range(24):
        url = mergir_url(date, h, cfg)
        out = cfg["RAW_DIR"] / f"merg_{date.year}{date.month:02d}{date.day:02d}{h:02d}_4km-pixel.nc4"

        if out.exists() and out.stat().st_size > 1_000_000:
            if verbose:
                print(f"  {date_str} hour {h:02d}: exists")
            continue

        if verbose:
            print(f"  {date_str} downloading hour {h:02d}/23 ...")

        cmd = [
            "curl.exe", "-L", "-n",
            "-b", cfg["COOKIES"], "-c", cfg["COOKIES"],
            "-o", str(out),
            url
        ]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"curl failed {r.returncode}\n{r.stderr}\nURL={url}")
        if out.stat().st_size < 1_000_000:
            raise RuntimeError(f"Downloaded file too small ({out.stat().st_size} bytes): {out}\nURL={url}")

        if verbose:
            print(f"  {date_str} hour {h:02d}: DONE ({out.stat().st_size/1024**2:.1f} MB)")

    print(f"[OK] downloaded raw MERGIR for {date_str}")

def raw_files_for_day(date_str, cfg=CFG):
    ymd = pd.Timestamp(date_str).strftime("%Y%m%d")
    return sorted(glob.glob(str(cfg["RAW_DIR"] / f"merg_{ymd}??_4km-pixel.nc4")))

def validate_raw_file(fp):
    try:
        ds = xr.open_dataset(fp, engine="netcdf4")
        ds.close()
        return True
    except Exception:
        return False

def download_mergir_hour(date_str, hour, cfg=CFG):
    date = pd.Timestamp(date_str).date()
    url = mergir_url(date, hour, cfg)
    out = cfg["RAW_DIR"] / f"merg_{date.year}{date.month:02d}{date.day:02d}{hour:02d}_4km-pixel.nc4"

    if out.exists():
        try:
            os.remove(out)
        except:
            pass

    print(f"  RE-DOWNLOAD {date_str} hour {hour:02d} ...")

    cmd = [
        "curl.exe", "-L", "-n",
        "-b", cfg["COOKIES"], "-c", cfg["COOKIES"],
        "-o", str(out),
        url
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"curl failed {r.returncode}\n{r.stderr}\nURL={url}")
    if out.stat().st_size < 1_000_000:
        raise RuntimeError(f"Downloaded file too small ({out.stat().st_size} bytes): {out}\nURL={url}")

    ok = validate_raw_file(str(out))
    print(f"  hour {hour:02d} size={out.stat().st_size/1024**2:.1f} MB valid={ok}")
    if not ok:
        raise RuntimeError(f"Re-downloaded file still not readable: {out}")

    return str(out)

def heal_day_raw(date_str, cfg=CFG):
    files = raw_files_for_day(date_str, cfg)
    if len(files) != 24:
        print(f"  heal_day_raw: file count is {len(files)} not 24; downloading full day...")
        download_mergir_day(date_str, cfg, verbose=False)
        files = raw_files_for_day(date_str, cfg)

    bad_hours = []
    for fp in files:
        hh = int(Path(fp).stem.split("_")[1][8:10])
        if not validate_raw_file(fp):
            bad_hours.append(hh)

    if bad_hours:
        print(f"  Found corrupted hours for {date_str}: {bad_hours}")
        for hh in bad_hours:
            download_mergir_hour(date_str, hh, cfg)
    else:
        print(f"  All raw files readable for {date_str}")

    return True

# ---------------------------
# SUBSET TO WNP
# ---------------------------
def subset_one_to_wnp(fp, cfg=CFG):
    ds = xr.open_dataset(fp, engine="netcdf4")

    latn = "lat" if "lat" in ds.coords else "latitude"
    lonn = "lon" if "lon" in ds.coords else "longitude"

    sub = ds.sel({
        latn: slice(cfg["LAT_MIN"], cfg["LAT_MAX"]),
        lonn: slice(cfg["LON_MIN"], cfg["LON_MAX"])
    })

    out = cfg["WNP_DIR"] / (Path(fp).stem + "_WNP.nc4")
    enc = {v: {"zlib": True, "complevel": 3, "dtype": "float32"} for v in sub.data_vars}
    sub.to_netcdf(out, encoding=enc)

    ds.close()
    sub.close()
    return str(out)

def subset_day_to_wnp(date_str, cfg=CFG):
    ymd = pd.Timestamp(date_str).strftime("%Y%m%d")
    raw_files = sorted(glob.glob(str(cfg["RAW_DIR"] / f"merg_{ymd}??_4km-pixel.nc4")))

    if len(raw_files) != 24:
        raise FileNotFoundError(f"Need 24 raw files; found {len(raw_files)} for {ymd}")

    new_count = 0
    for fp in raw_files:
        out = cfg["WNP_DIR"] / (Path(fp).stem + "_WNP.nc4")
        if out.exists() and out.stat().st_size > 100_000:
            continue
        subset_one_to_wnp(fp, cfg)
        new_count += 1

    wnp_files = glob.glob(str(cfg["WNP_DIR"] / f"merg_{ymd}??_4km-pixel_WNP.nc4"))
    print(f"[OK] subset WNP for {date_str}. new={new_count}, wnp_hours={len(wnp_files)}")

# ---------------------------
# 30-MIN -> 3-HOURLY
# ---------------------------
def snap_time_to_30min(ds):
    t = pd.to_datetime(ds["time"].values).round("30min")
    return ds.assign_coords(time=("time", t))

def open_day_wnp_30min(date_str, cfg=CFG):
    ymd = pd.Timestamp(date_str).strftime("%Y%m%d")
    files = sorted(glob.glob(str(cfg["WNP_DIR"] / f"merg_{ymd}??_4km-pixel_WNP.nc4")))

    if len(files) != 24:
        raise FileNotFoundError(f"Need 24 WNP hourly files; found {len(files)} for {ymd}")

    dsets = []
    for fp in files:
        ds = xr.open_dataset(fp, engine="netcdf4")[["Tb"]]
        ds = snap_time_to_30min(ds)
        dsets.append(ds)

    day = xr.concat(dsets, dim="time").sortby("time")
    t = pd.to_datetime(day.time.values)
    _, idx = np.unique(t.values, return_index=True)
    day = day.isel(time=np.sort(idx))
    return day

def make_day_3hourly(date_str, cfg=CFG):
    day_30m = open_day_wnp_30min(date_str, cfg)
    day_3h = day_30m.resample(time="3h").nearest()
    return day_30m, day_3h

# ---------------------------
# MCS OBJECT DETECTION
# ---------------------------
def pixel_area_km2(lat_vals, lon_vals):
    R = 6371.0
    lat = np.deg2rad(lat_vals)
    dlat = np.abs(np.deg2rad(lat_vals[1] - lat_vals[0]))
    dlon = np.abs(np.deg2rad(lon_vals[1] - lon_vals[0]))
    return (R**2) * dlat * dlon * np.cos(lat)

def detect_objects_one_time(Tb2d, lat_vals, lon_vals, tb_th, area_min_km2):
    mask = np.isfinite(Tb2d) & (Tb2d < tb_th)
    lab, nlab = label(mask, structure=np.ones((3, 3), dtype=int))

    if nlab == 0:
        return []

    arow = pixel_area_km2(lat_vals, lon_vals)
    nlon = len(lon_vals)
    objs = []

    for oid in range(1, nlab + 1):
        yy, xx = np.where(lab == oid)
        if len(yy) == 0:
            continue

        area = float(arow[yy].sum())
        if area < area_min_km2:
            continue

        tb = Tb2d[yy, xx]
        w = arow[yy]
        clat = float(np.sum(lat_vals[yy] * w) / np.sum(w))
        clon = float(np.sum(lon_vals[xx] * w) / np.sum(w))
        lin = (yy.astype(np.int64) * nlon + xx.astype(np.int64))

        objs.append({
            "area_km2": area,
            "centroid_lat": clat,
            "centroid_lon": clon,
            "tb_mean": float(np.nanmean(tb)),
            "tb_min": float(np.nanmin(tb)),
            "npix": int(len(yy)),
            "linpix": set(lin.tolist()),
        })

    return objs

def detect_objects_day(day_3h, cfg=CFG):
    lat_vals = day_3h["lat"].values
    lon_vals = day_3h["lon"].values

    rows = []
    pix_index = {}

    for tt in day_3h.time.values:
        Tb2d = day_3h["Tb"].sel(time=tt).values

        objs = detect_objects_one_time(
            Tb2d, lat_vals, lon_vals,
            tb_th=cfg["TB_TH"],
            area_min_km2=cfg["AREA_MIN_KM2"]
        )

        for j, ob in enumerate(objs):
            obj_key = f"{pd.Timestamp(tt):%Y%m%d%H%M}_{j:04d}"
            pix_index[(pd.Timestamp(tt), obj_key)] = ob["linpix"]

            rows.append({
                "time": pd.Timestamp(tt),
                "obj_key": obj_key,
                "area_km2": ob["area_km2"],
                "centroid_lat": ob["centroid_lat"],
                "centroid_lon": ob["centroid_lon"],
                "tb_mean": ob["tb_mean"],
                "tb_min": ob["tb_min"],
                "npix": ob["npix"],
            })

    objects_df = pd.DataFrame(rows).sort_values(["time", "obj_key"]).reset_index(drop=True)
    return objects_df, pix_index

# ---------------------------
# TRACKING + MOTION
# ---------------------------
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.deg2rad, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.deg2rad, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1)*np.sin(lat2) - np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
    brng = np.degrees(np.arctan2(x, y))
    return (brng + 360) % 360

def track_objects(objects_df, pix_index, min_iou=0.01, max_km=300.0):
    df = objects_df.copy()
    df["track_id"] = None

    times = sorted(df["time"].unique())
    next_tid = 1

    for idx in df[df["time"] == times[0]].index:
        df.loc[idx, "track_id"] = next_tid
        next_tid += 1

    for k in range(len(times) - 1):
        ta, tb = times[k], times[k + 1]
        A = df[df["time"] == ta]
        B = df[df["time"] == tb]

        unmatched_A = set(A.index.tolist())
        B_order = B.sort_values("area_km2", ascending=False).index.tolist()

        for b_idx in B_order:
            b = df.loc[b_idx]
            bset = pix_index.get((pd.Timestamp(tb), b["obj_key"]), None)
            if bset is None:
                continue

            best_a = None
            best_iou = -1.0

            for a_idx in list(unmatched_A):
                a = df.loc[a_idx]
                aset = pix_index.get((pd.Timestamp(ta), a["obj_key"]), None)
                if aset is None:
                    continue

                inter = len(aset & bset)
                if inter == 0:
                    continue

                union = len(aset | bset)
                iou = inter / union
                if iou > best_iou:
                    best_iou = iou
                    best_a = a_idx

            if best_a is not None and best_iou >= min_iou:
                df.loc[b_idx, "track_id"] = int(df.loc[best_a, "track_id"])
                unmatched_A.remove(best_a)
                continue

            dmin = None
            amin = None
            for a_idx in list(unmatched_A):
                a = df.loc[a_idx]
                d = float(haversine_km(
                    a["centroid_lat"], a["centroid_lon"],
                    b["centroid_lat"], b["centroid_lon"]
                ))
                if dmin is None or d < dmin:
                    dmin = d
                    amin = a_idx

            if amin is not None and dmin is not None and dmin <= max_km:
                df.loc[b_idx, "track_id"] = int(df.loc[amin, "track_id"])
                unmatched_A.remove(amin)
                continue

            df.loc[b_idx, "track_id"] = next_tid
            next_tid += 1

    df = df.sort_values(["track_id", "time"]).reset_index(drop=True)

    lat_prev = df.groupby("track_id")["centroid_lat"].shift(1)
    lon_prev = df.groupby("track_id")["centroid_lon"].shift(1)
    t_prev   = df.groupby("track_id")["time"].shift(1)

    df["dt_h"] = (df["time"] - t_prev).dt.total_seconds() / 3600.0
    df["dist_km"] = haversine_km(
        lat_prev.values, lon_prev.values,
        df["centroid_lat"].values, df["centroid_lon"].values
    )
    df["speed_kmh"] = df["dist_km"] / df["dt_h"]
    df["direction_deg"] = bearing_deg(
        lat_prev.values, lon_prev.values,
        df["centroid_lat"].values, df["centroid_lon"].values
    )

    return df

# ---------------------------
# CLEANUP
# ---------------------------
def cleanup_day_files(date_str, cfg=CFG, delete_raw=True, delete_wnp=False):
    ymd = pd.Timestamp(date_str).strftime("%Y%m%d")

    if delete_raw:
        raw_files = glob.glob(str(cfg["RAW_DIR"] / f"merg_{ymd}??_4km-pixel.nc4"))
        for fp in raw_files:
            try:
                os.remove(fp)
            except Exception as e:
                print(f"[WARN] could not delete raw file: {fp} -> {e}")

    if delete_wnp:
        wnp_files = glob.glob(str(cfg["WNP_DIR"] / f"merg_{ymd}??_4km-pixel_WNP.nc4"))
        for fp in wnp_files:
            try:
                os.remove(fp)
            except Exception as e:
                print(f"[WARN] could not delete WNP file: {fp} -> {e}")

# ---------------------------
# ONE-DAY RUNNER
# ---------------------------
def run_one_day(date_str, cfg=CFG, do_download=False, cleanup_raw=True, cleanup_wnp=False):
    print(f"\n=== RUN DAY: {date_str} ===")

    if do_download:
        download_mergir_day(date_str, cfg, verbose=True)

    subset_day_to_wnp(date_str, cfg)

    day_30m, day_3h = make_day_3hourly(date_str, cfg)
    print("30-min ntimes:", day_30m.sizes["time"])
    print("3-hour ntimes:", day_3h.sizes["time"])

    objects_df, pix_index = detect_objects_day(day_3h, cfg)
    tracks_df = track_objects(
        objects_df, pix_index,
        min_iou=cfg["TRACK_MIN_IOU"],
        max_km=cfg["TRACK_MAX_KM"]
    )

    ymd = pd.Timestamp(date_str).strftime("%Y%m%d")
    objects_csv = cfg["OUT_DIR"] / f"mcs_objects_3h_{ymd}.csv"
    tracks_csv  = cfg["OUT_DIR"] / f"mcs_tracks_3h_{ymd}.csv"

    objects_df.to_csv(objects_csv, index=False)
    tracks_df.to_csv(tracks_csv, index=False)

    print("[SAVED]", objects_csv.name, "| rows:", len(objects_df))
    print("[SAVED]", tracks_csv.name,  "| rows:", len(tracks_df))
    print("n tracks:", tracks_df["track_id"].nunique())

    try:
        day_30m.close()
    except:
        pass
    try:
        day_3h.close()
    except:
        pass

    if (cleanup_raw or cleanup_wnp) and objects_csv.exists() and tracks_csv.exists():
        cleanup_day_files(
            date_str,
            cfg=cfg,
            delete_raw=cleanup_raw,
            delete_wnp=cleanup_wnp
        )
        print(f"[CLEANUP] raw={cleanup_raw}, wnp={cleanup_wnp} for {date_str}")

    return objects_df, tracks_df

# ---------------------------
# MONTHLY CONCAT
# ---------------------------
def concat_month(year, month, cfg=CFG):
    ym = f"{year}{month:02d}"

    out_tracks  = cfg["OUT_DIR"] / f"mcs_tracks_3h_{ym}.csv"
    out_objects = cfg["OUT_DIR"] / f"mcs_objects_3h_{ym}.csv"

    track_files  = sorted(glob.glob(str(cfg["OUT_DIR"] / f"mcs_tracks_3h_{ym}??.csv")))
    object_files = sorted(glob.glob(str(cfg["OUT_DIR"] / f"mcs_objects_3h_{ym}??.csv")))

    if len(track_files) == 0 or len(object_files) == 0:
        raise FileNotFoundError(f"Missing daily files for {ym}. tracks={len(track_files)} objects={len(object_files)}")

    tracks = pd.concat([pd.read_csv(f) for f in track_files], ignore_index=True)
    objs   = pd.concat([pd.read_csv(f) for f in object_files], ignore_index=True)

    tracks["time"] = pd.to_datetime(tracks["time"])
    objs["time"]   = pd.to_datetime(objs["time"])

    tracks["date"] = tracks["time"].dt.strftime("%Y%m%d")
    tracks["track_id_month"] = tracks["date"] + "_" + tracks["track_id"].astype(int).astype(str)

    tracks.to_csv(out_tracks, index=False)
    objs.to_csv(out_objects, index=False)

    print(f"[MONTH SAVED] {out_tracks.name} rows={len(tracks)} unique_tracks={tracks['track_id_month'].nunique()}")
    print(f"[MONTH SAVED] {out_objects.name} rows={len(objs)}")

    return out_tracks, out_objects

# ---------------------------
# MONTH RUNNER
# ---------------------------
def run_month_and_concat(
    year,
    month,
    download_missing=True,
    cleanup_raw_after_day=True,
    cleanup_wnp_after_day=False,
    cfg=CFG
):
    start = pd.Timestamp(year=year, month=month, day=1)
    end = (start + pd.offsets.MonthEnd(1)).normalize()
    ndays = calendar.monthrange(year, month)[1]

    print(f"\n==============================")
    print(f"MONTH: {year}-{month:02d}  (days={ndays})")
    print(f"==============================")

    for i, day in enumerate(pd.date_range(start, end, freq="D"), start=1):
        ds = day.strftime("%Y-%m-%d")
        ymd = day.strftime("%Y%m%d")

        out_daily = cfg["OUT_DIR"] / f"mcs_tracks_3h_{ymd}.csv"
        if out_daily.exists() and out_daily.stat().st_size > 10_000:
            print(f"[{i:02d}/{ndays}] SKIP processed: {ds}")
            continue

        raw_count = len(glob.glob(str(cfg["RAW_DIR"] / f"merg_{ymd}??_4km-pixel.nc4")))
        print(f"[{i:02d}/{ndays}] START {ds} | raw={raw_count}/24")

        try:
            if download_missing and raw_count != 24:
                download_mergir_day(ds, cfg, verbose=True)

            try:
                heal_day_raw(ds, cfg)
                run_one_day(
                    ds,
                    cfg,
                    do_download=False,
                    cleanup_raw=cleanup_raw_after_day,
                    cleanup_wnp=cleanup_wnp_after_day
                )
            except Exception as e1:
                print("   processing failed once; heal+retry:", e1)
                heal_day_raw(ds, cfg)
                run_one_day(
                    ds,
                    cfg,
                    do_download=False,
                    cleanup_raw=cleanup_raw_after_day,
                    cleanup_wnp=cleanup_wnp_after_day
                )

            print(f"[{i:02d}/{ndays}] DONE  {ds}")

        except Exception as e:
            print(f"[{i:02d}/{ndays}] FAILED {ds} -> {e}")

    print(f"\n=== CONCAT MONTH {year}-{month:02d} ===")
    concat_month(year, month, cfg)
    print(f"=== MONTH FINISHED {year}-{month:02d} ===")

# ---------------------------
# MAIN RUN
# ---------------------------
YEARS = list(range(2009, 2026))
MONTHS = [6, 7, 8, 9, 10]

for y in YEARS:
    for m in MONTHS:
        print("\n" + "=" * 70)
        print(f"RUNNING MCS PIPELINE: {y}-{m:02d}")
        print("=" * 70)

        run_month_and_concat(
            y, m,
            download_missing=True,
            cleanup_raw_after_day=True,   # SAFE: delete RAW after successful daily CSV save
            cleanup_wnp_after_day=False   # keep WNP for now
        )


RUNNING MCS PIPELINE: 2009-06

MONTH: 2009-06  (days=30)
[01/30] SKIP processed: 2009-06-01
[02/30] SKIP processed: 2009-06-02
[03/30] SKIP processed: 2009-06-03
[04/30] SKIP processed: 2009-06-04
[05/30] SKIP processed: 2009-06-05
[06/30] SKIP processed: 2009-06-06
[07/30] SKIP processed: 2009-06-07
[08/30] SKIP processed: 2009-06-08
[09/30] SKIP processed: 2009-06-09
[10/30] SKIP processed: 2009-06-10
[11/30] SKIP processed: 2009-06-11
[12/30] SKIP processed: 2009-06-12
[13/30] SKIP processed: 2009-06-13
[14/30] SKIP processed: 2009-06-14
[15/30] SKIP processed: 2009-06-15
[16/30] SKIP processed: 2009-06-16
[17/30] SKIP processed: 2009-06-17
[18/30] SKIP processed: 2009-06-18
[19/30] SKIP processed: 2009-06-19
[20/30] SKIP processed: 2009-06-20
[21/30] SKIP processed: 2009-06-21
[22/30] SKIP processed: 2009-06-22
[23/30] SKIP processed: 2009-06-23
[24/30] SKIP processed: 2009-06-24
[25/30] SKIP processed: 2009-06-25
[26/30] SKIP processed: 2009-06-26
[27/30] SKIP processed: 2009-06-

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import glob
import pandas as pd

OUT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output")

monthly = sorted(glob.glob(str(OUT_DIR / "mcs_tracks_3h_*.csv")))

# only monthly files, not daily files
monthly = [f for f in monthly if len(Path(f).stem.split("_")[-1]) == 6]

print("monthly files:", len(monthly))
print(monthly[:3])
print(monthly[-3:])

monthly files: 87
['C:\\Users\\_s2218026\\Documents\\lab\\mcs_output\\mcs_tracks_3h_200906.csv', 'C:\\Users\\_s2218026\\Documents\\lab\\mcs_output\\mcs_tracks_3h_200907.csv', 'C:\\Users\\_s2218026\\Documents\\lab\\mcs_output\\mcs_tracks_3h_200908.csv']
['C:\\Users\\_s2218026\\Documents\\lab\\mcs_output\\mcs_tracks_3h_202508.csv', 'C:\\Users\\_s2218026\\Documents\\lab\\mcs_output\\mcs_tracks_3h_202509.csv', 'C:\\Users\\_s2218026\\Documents\\lab\\mcs_output\\mcs_tracks_3h_202510.csv']


In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output")

YEARS = list(range(2009, 2026))
MONTHS = [6, 7, 8, 9, 10]   # keep consistent everywhere

dfs = []
missing = []

for y in YEARS:
    for m in MONTHS:
        ym = f"{y}{m:02d}"
        fp = OUT_DIR / f"mcs_tracks_3h_{ym}.csv"

        if fp.exists():
            df = pd.read_csv(fp)
            dfs.append(df)
            print("Loaded:", fp.name, "rows =", len(df))
        else:
            missing.append(fp.name)
            print("Missing:", fp.name)

if len(dfs) == 0:
    raise ValueError("No monthly MCS track files were found.")

all_tracks = pd.concat(dfs, ignore_index=True)

out_fp = OUT_DIR / "mcs_tracks_3h_2009_2025_JJASO.csv"
all_tracks.to_csv(out_fp, index=False)

print("\nSaved:", out_fp)
print("Total rows:", len(all_tracks))

if "time" in all_tracks.columns:
    print("\nCounts by month:")
    print(all_tracks["time"].astype(str).str[:7].value_counts().sort_index())

print("\nMissing monthly files:", len(missing))

print("\nColumns:")
print(all_tracks.columns.tolist())

print("\nHead:")
print(all_tracks.head())

Loaded: mcs_tracks_3h_200906.csv rows = 8118
Loaded: mcs_tracks_3h_200907.csv rows = 10679
Loaded: mcs_tracks_3h_200908.csv rows = 8771
Loaded: mcs_tracks_3h_200909.csv rows = 9535
Loaded: mcs_tracks_3h_200910.csv rows = 7334
Loaded: mcs_tracks_3h_201006.csv rows = 8333
Loaded: mcs_tracks_3h_201007.csv rows = 10337
Loaded: mcs_tracks_3h_201008.csv rows = 10573
Loaded: mcs_tracks_3h_201009.csv rows = 9009
Loaded: mcs_tracks_3h_201010.csv rows = 9163
Loaded: mcs_tracks_3h_201106.csv rows = 8695
Loaded: mcs_tracks_3h_201107.csv rows = 10092
Loaded: mcs_tracks_3h_201108.csv rows = 10321
Loaded: mcs_tracks_3h_201109.csv rows = 10328
Loaded: mcs_tracks_3h_201110.csv rows = 8146
Loaded: mcs_tracks_3h_201206.csv rows = 8231
Loaded: mcs_tracks_3h_201207.csv rows = 11189
Loaded: mcs_tracks_3h_201208.csv rows = 9887
Loaded: mcs_tracks_3h_201209.csv rows = 10038
Loaded: mcs_tracks_3h_201210.csv rows = 6809
Loaded: mcs_tracks_3h_201306.csv rows = 8039
Loaded: mcs_tracks_3h_201307.csv rows = 10339
L

In [ ]:
YEARS = list(range(2009, 2018))
MONTHS = [6, 7, 8, 9, 10]

for y in YEARS:
    for m in MONTHS:
        print("\n" + "="*70)
        print(f"RUNNING MCS PIPELINE: {y}-{m:02d}")
        print("="*70)
        run_month_and_concat(y, m, download_missing=True)


RUNNING MCS PIPELINE: 2009-06

MONTH: 2009-06  (days=30)
[01/30] SKIP processed: 2009-06-01
[02/30] SKIP processed: 2009-06-02
[03/30] SKIP processed: 2009-06-03
[04/30] SKIP processed: 2009-06-04
[05/30] SKIP processed: 2009-06-05
[06/30] SKIP processed: 2009-06-06
[07/30] SKIP processed: 2009-06-07
[08/30] SKIP processed: 2009-06-08
[09/30] SKIP processed: 2009-06-09
[10/30] SKIP processed: 2009-06-10
[11/30] SKIP processed: 2009-06-11
[12/30] SKIP processed: 2009-06-12
[13/30] SKIP processed: 2009-06-13
[14/30] SKIP processed: 2009-06-14
[15/30] SKIP processed: 2009-06-15
[16/30] SKIP processed: 2009-06-16
[17/30] SKIP processed: 2009-06-17
[18/30] SKIP processed: 2009-06-18
[19/30] SKIP processed: 2009-06-19
[20/30] SKIP processed: 2009-06-20
[21/30] SKIP processed: 2009-06-21
[22/30] SKIP processed: 2009-06-22
[23/30] SKIP processed: 2009-06-23
[24/30] SKIP processed: 2009-06-24
[25/30] SKIP processed: 2009-06-25
[26/30] SKIP processed: 2009-06-26
[27/30] SKIP processed: 2009-06-

In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output")

YEARS = list(range(2009, 2018))
MONTHS = [6, 7, 8, 9, 10]

dfs = []
missing = []

for y in YEARS:
    for m in MONTHS:
        ym = f"{y}{m:02d}"
        fp = OUT_DIR / f"mcs_tracks_3h_{ym}.csv"

        if fp.exists():
            df = pd.read_csv(fp)
            dfs.append(df)
            print("Loaded:", fp.name, "rows =", len(df))
        else:
            missing.append(fp.name)
            print("Missing:", fp.name)

if len(dfs) == 0:
    raise ValueError("No monthly MCS track files were found.")

all_tracks = pd.concat(dfs, ignore_index=True)

out_fp = OUT_DIR / "mcs_tracks_3h_2009_2017_JJASO.csv"
all_tracks.to_csv(out_fp, index=False)

print("\nSaved:", out_fp)
print("Total rows:", len(all_tracks))

if "time" in all_tracks.columns:
    print("\nCounts by month:")
    print(all_tracks["time"].astype(str).str[:7].value_counts().sort_index())

print("\nMissing monthly files:", len(missing))
print("\nColumns:")
print(all_tracks.columns.tolist())
print("\nHead:")
print(all_tracks.head())

Loaded: mcs_tracks_3h_200906.csv rows = 8118
Loaded: mcs_tracks_3h_200907.csv rows = 10679
Loaded: mcs_tracks_3h_200908.csv rows = 8771
Loaded: mcs_tracks_3h_200909.csv rows = 9535
Loaded: mcs_tracks_3h_200910.csv rows = 7334
Loaded: mcs_tracks_3h_201006.csv rows = 8333
Loaded: mcs_tracks_3h_201007.csv rows = 10337
Loaded: mcs_tracks_3h_201008.csv rows = 10573
Loaded: mcs_tracks_3h_201009.csv rows = 9009
Loaded: mcs_tracks_3h_201010.csv rows = 9163
Loaded: mcs_tracks_3h_201106.csv rows = 8695
Loaded: mcs_tracks_3h_201107.csv rows = 10092
Loaded: mcs_tracks_3h_201108.csv rows = 10321
Loaded: mcs_tracks_3h_201109.csv rows = 10328
Loaded: mcs_tracks_3h_201110.csv rows = 8146
Loaded: mcs_tracks_3h_201206.csv rows = 8231
Loaded: mcs_tracks_3h_201207.csv rows = 11189
Loaded: mcs_tracks_3h_201208.csv rows = 9887
Loaded: mcs_tracks_3h_201209.csv rows = 10038
Loaded: mcs_tracks_3h_201210.csv rows = 6809
Loaded: mcs_tracks_3h_201306.csv rows = 8039
Loaded: mcs_tracks_3h_201307.csv rows = 10339
L

In [ ]:
# ============================================================
# PLOT MERGIR IR + MCS DETECTION LOGIC
# Raw IR -> Tb mask -> old objects -> NEW soft objects
# ============================================================

from pathlib import Path
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from scipy.ndimage import label, binary_closing, binary_dilation, generate_binary_structure

# ============================================================
# CONFIG
# ============================================================

WNP_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mergir_wnp")
OUT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest\figures_method")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Choose one case
DATE_STR = "2026-05-17"
TARGET_TIME = "2026-05-17 03:00"   # nearest available time will be used

# Domain
LAT_MIN, LAT_MAX = 0, 30
LON_MIN, LON_MAX = 100, 180

# MCS threshold
TB_TH = 233.0
AREA_MIN_KM2 = 5000.0

# NEW / soft method settings
USE_CLOSING = True
CLOSE_ITERS = 1
MERGE_ITERS = 2

STRUCT2D = generate_binary_structure(2, 2)  # 8-connected


# ============================================================
# HELPERS
# ============================================================

def pixel_area_km2(lat_vals, lon_vals):
    """
    Approximate pixel area for regular lat-lon grid.
    Returns area per latitude row.
    """
    R = 6371.0
    lat_vals = np.asarray(lat_vals, dtype=float)
    lon_vals = np.asarray(lon_vals, dtype=float)

    dlat = np.abs(np.deg2rad(lat_vals[1] - lat_vals[0]))
    dlon = np.abs(np.deg2rad(lon_vals[1] - lon_vals[0]))
    lat_rad = np.deg2rad(lat_vals)

    return (R ** 2) * dlat * dlon * np.cos(lat_rad)


def open_day_wnp(date_str):
    """
    Opens 24 WNP-cropped MERGIR hourly files:
    merg_YYYYMMDDHH_4km-pixel_WNP.nc4
    """
    ymd = pd.Timestamp(date_str).strftime("%Y%m%d")
    files = sorted(glob.glob(str(WNP_DIR / f"merg_{ymd}??_4km-pixel_WNP.nc4")))

    if len(files) == 0:
        raise FileNotFoundError(f"No MERGIR WNP files found for {ymd} in:\n{WNP_DIR}")

    print(f"Found {len(files)} files for {ymd}")

    dsets = []
    for fp in files:
        ds = xr.open_dataset(fp, engine="netcdf4")[["Tb"]]
        dsets.append(ds)

    day = xr.concat(dsets, dim="time").sortby("time")

    # Remove duplicate timestamps if any
    t = pd.to_datetime(day.time.values)
    _, idx = np.unique(t.values, return_index=True)
    day = day.isel(time=np.sort(idx))

    return day


def get_lat_lon_names(ds):
    lat_candidates = ["lat", "latitude", "Lat", "Latitude"]
    lon_candidates = ["lon", "longitude", "Lon", "Longitude"]

    lat_name = next((x for x in lat_candidates if x in ds.coords or x in ds.dims), None)
    lon_name = next((x for x in lon_candidates if x in ds.coords or x in ds.dims), None)

    if lat_name is None or lon_name is None:
        raise ValueError(f"Could not identify lat/lon names. Dataset dims: {ds.dims}, coords: {list(ds.coords)}")

    return lat_name, lon_name


def filter_objects_by_area(labels_raw, tb, lat_vals, lon_vals, area_min_km2):
    """
    Keeps only connected components whose original cold-pixel area >= area_min_km2.
    Returns filtered label array and object table.
    """
    arow = pixel_area_km2(lat_vals, lon_vals)
    labels_filtered = np.zeros_like(labels_raw, dtype=np.int32)

    rows = []
    new_id = 1

    for oid in range(1, int(labels_raw.max()) + 1):
        mask = labels_raw == oid
        yy, xx = np.where(mask)

        if len(yy) == 0:
            continue

        area = float(arow[yy].sum())

        if area < area_min_km2:
            continue

        tb_obj = tb[yy, xx]
        w = arow[yy]

        clat = float(np.sum(lat_vals[yy] * w) / np.sum(w))
        clon = float(np.sum(lon_vals[xx] * w) / np.sum(w))

        labels_filtered[mask] = new_id

        rows.append({
            "object_id": new_id,
            "area_km2": area,
            "centroid_lat": clat,
            "centroid_lon": clon,
            "tb_mean": float(np.nanmean(tb_obj)),
            "tb_min": float(np.nanmin(tb_obj)),
            "n_pixels": len(yy),
        })

        new_id += 1

    return labels_filtered, pd.DataFrame(rows)


def detect_old_objects(tb, lat_vals, lon_vals):
    """
    OLD method:
    Tb < 233 K -> connected components -> area filter
    """
    cold_mask = np.isfinite(tb) & (tb < TB_TH)
    labels_raw, nlab = label(cold_mask, structure=STRUCT2D)
    labels_filtered, obj_df = filter_objects_by_area(labels_raw, tb, lat_vals, lon_vals, AREA_MIN_KM2)

    return cold_mask, labels_filtered, obj_df


def detect_soft_objects(tb, lat_vals, lon_vals):
    """
    NEW / soft method:
    1. Original cold mask: Tb < 233 K
    2. Closing fills small gaps
    3. Dilation merges nearby fragments
    4. Connected components on modified mask
    5. Statistics taken from original cold pixels only
    """
    orig_mask = np.isfinite(tb) & (tb < TB_TH)
    work_mask = orig_mask.copy()

    if USE_CLOSING:
        for _ in range(CLOSE_ITERS):
            work_mask = binary_closing(work_mask, structure=STRUCT2D)

    for _ in range(MERGE_ITERS):
        work_mask = binary_dilation(work_mask, structure=STRUCT2D)

    labels_raw, nlab = label(work_mask, structure=STRUCT2D)

    # Important: keep only original cold pixels inside each soft merged object
    member_labels = np.zeros_like(labels_raw, dtype=np.int32)

    for oid in range(1, nlab + 1):
        merged_region = labels_raw == oid
        member_mask = orig_mask & merged_region
        member_labels[member_mask] = oid

    labels_filtered, obj_df = filter_objects_by_area(member_labels, tb, lat_vals, lon_vals, AREA_MIN_KM2)

    return orig_mask, work_mask, labels_filtered, obj_df


# ============================================================
# LOAD DATA
# ============================================================

day = open_day_wnp(DATE_STR)
lat_name, lon_name = get_lat_lon_names(day)

# Select nearest target time
target = pd.Timestamp(TARGET_TIME)
snap = day.sel(time=target, method="nearest")

actual_time = pd.Timestamp(snap.time.values)
print("Requested time:", target)
print("Actual MERGIR time used:", actual_time)

# Crop domain if needed
snap = snap.sel({
    lat_name: slice(LAT_MIN, LAT_MAX),
    lon_name: slice(LON_MIN, LON_MAX)
})

tb = snap["Tb"].values
lat_vals = snap[lat_name].values
lon_vals = snap[lon_name].values

# If latitude is descending, flip it for normal plotting
if lat_vals[0] > lat_vals[-1]:
    lat_vals = lat_vals[::-1]
    tb = tb[::-1, :]

print("Tb shape:", tb.shape)
print("Lat range:", lat_vals.min(), lat_vals.max())
print("Lon range:", lon_vals.min(), lon_vals.max())


# ============================================================
# DETECT OBJECTS
# ============================================================

cold_old, labels_old, obj_old = detect_old_objects(tb, lat_vals, lon_vals)
cold_soft, work_soft, labels_soft, obj_soft = detect_soft_objects(tb, lat_vals, lon_vals)

print("\nOLD objects:", len(obj_old))
display(obj_old.sort_values("area_km2", ascending=False).head(10))

print("\nNEW soft objects:", len(obj_soft))
display(obj_soft.sort_values("area_km2", ascending=False).head(10))


# ============================================================
# PLOT
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(24, 6), constrained_layout=True)

# Panel 1: Raw IR
im0 = axes[0].pcolormesh(lon_vals, lat_vals, tb, shading="auto", cmap="gray_r", vmin=190, vmax=300)
axes[0].contour(lon_vals, lat_vals, cold_old, levels=[0.5], linewidths=0.8)
axes[0].set_title("(a) Raw MERGIR IR Brightness Temperature")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
cb0 = plt.colorbar(im0, ax=axes[0])
cb0.set_label("Tb (K)")

# Panel 2: Cold cloud mask
im1 = axes[1].pcolormesh(lon_vals, lat_vals, cold_old.astype(int), shading="auto", cmap="gray_r")
axes[1].set_title(f"(b) Cold Cloud Mask\nTb < {TB_TH:.0f} K")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")

# Panel 3: OLD objects
masked_old = np.ma.masked_where(labels_old == 0, labels_old)
im2 = axes[2].pcolormesh(lon_vals, lat_vals, masked_old, shading="auto", cmap="tab20")
axes[2].contour(lon_vals, lat_vals, labels_old > 0, levels=[0.5], linewidths=0.8)
axes[2].set_title(f"(c) OLD Connected Objects\nArea ≥ {AREA_MIN_KM2:.0f} km²")
axes[2].set_xlabel("Longitude")
axes[2].set_ylabel("Latitude")

# Add object IDs
for _, r in obj_old.iterrows():
    axes[2].text(r["centroid_lon"], r["centroid_lat"], str(int(r["object_id"])),
                 fontsize=8, ha="center", va="center")

# Panel 4: NEW soft objects
masked_soft = np.ma.masked_where(labels_soft == 0, labels_soft)
im3 = axes[3].pcolormesh(lon_vals, lat_vals, masked_soft, shading="auto", cmap="tab20")
axes[3].contour(lon_vals, lat_vals, labels_soft > 0, levels=[0.5], linewidths=0.8)
axes[3].set_title(f"(d) NEW Soft Objects\nClosing + Dilation")
axes[3].set_xlabel("Longitude")
axes[3].set_ylabel("Latitude")

# Add object IDs
for _, r in obj_soft.iterrows():
    axes[3].text(r["centroid_lon"], r["centroid_lat"], str(int(r["object_id"])),
                 fontsize=8, ha="center", va="center")

for ax in axes:
    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

fig.suptitle(
    f"MERGIR-based MCS Detection Example | {actual_time:%Y-%m-%d %H:%M UTC}",
    fontsize=16
)

out_png = OUT_DIR / f"mergir_mcs_detection_logic_{actual_time:%Y%m%d_%H%M}.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved figure:")
print(out_png)

FileNotFoundError: No MERGIR WNP files found for 20260517 in:
C:\Users\_s2218026\Documents\lab\mergir_wnp